# LLM assistant (RAG chatbot)

A chatbot or assistant that answers questions using your own documents and data.

Two ways to start, from Hello Model: https://sayhellomodel.com/models/llm-rag/

**Your data:** the code uses example file names (such as `data.csv`). Upload your own file with the Files panel on the left, or change the name in the code. Run the cells from the top.

## Quick start

Hosted LLM API (Claude) + managed RAG / long context: No infrastructure to run; small corpora can even fit in the context window.

In [ ]:
%pip install -q anthropic pypdf

In [ ]:
# pip install anthropic pypdf
import anthropic
from pypdf import PdfReader

text = "\n".join(p.extract_text() for p in PdfReader("manual.pdf").pages)
client = anthropic.Anthropic()
resp = client.messages.create(
    model="claude-sonnet-5-5", max_tokens=1024,
    system="Answer only from the provided manual. If unsure, say you don't know.",
    messages=[{"role": "user", "content": f"<manual>{text}</manual>\n\nHow do I reset the device?"}])
print(resp.content[0].text)

## Train your own model

RAG: hosted LLM + embeddings + vector DB: Scales to thousands of documents, answers cite sources, data stays updatable without retraining.

In [ ]:
%pip install -q anthropic chromadb sentence-transformers

In [ ]:
# pip install anthropic chromadb sentence-transformers
import anthropic, chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction("all-MiniLM-L6-v2")
db = chromadb.PersistentClient("./index").get_or_create_collection("docs", embedding_function=ef)

# 1) Index: chunks = list of (id, text, source)
# db.add(ids=[c[0] for c in chunks], documents=[c[1] for c in chunks],
#        metadatas=[{"source": c[2]} for c in chunks])

# 2) Retrieve + generate
def answer(question: str) -> str:
    hits = db.query(query_texts=[question], n_results=5)
    context = "\n\n".join(f"[{m['source']}] {d}" for d, m in
                           zip(hits["documents"][0], hits["metadatas"][0]))
    msg = anthropic.Anthropic().messages.create(
        model="claude-sonnet-5-5", max_tokens=800,
        system="Answer using only the context. Cite sources in [brackets].",
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    return msg.content[0].text

print(answer("What is the refund policy?"))